In [ ]:
# ============================================================
# Cell 1. Imports and global parameters
# ============================================================

import os
import json
import warnings
import inspect
import traceback

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

from bayesian_gp_cvloss.optimizer import GPCrossValidatedOptimizer


# ============================================================
# 1. File paths
# ============================================================

# Existing experimental dataset: 15 measured formulations
SEED_FILE = 'model-data/SEED.csv'

# Independent candidate formulation space
CANDIDATE_FILE = 'model-data/design_space.csv'

# Result directory
RESULT_DIR = 'results/GP-UCB'

os.makedirs(RESULT_DIR, exist_ok=True)


# ============================================================
# 2. Data columns
# ============================================================

FEATURE_NAMES = [
    'Al/Fe',
    '(Al+Fe)/Si',
    '(Al+Fe)/OH'
]

TARGET_NAME = 'RE'


# ============================================================
# 3. Repeated-run settings
# ============================================================

N_SEEDS = 30

SEED_LIST = list(range(N_SEEDS))

INITIAL_TRAIN_SIZE = 12

N_SPLITS = 5


# ============================================================
# 4. GP model parameters
# ============================================================

MAX_EVALS = 50


# ============================================================
# 5. Batch acquisition parameters
# ============================================================

TOP_N = 5

KAPPA = 0.5

DIVERSITY_SCALE = 0.6

LAMBDA_PEN = 0.4

MIN_DISTANCE_FRAC = 0.02

CLIP_PERCENTILES = (2, 98)

SIGMA_CAP_MULTIPLIER = 3.0


# ============================================================
# 6. Output control
# ============================================================

# False: stop if existing seed results are found
# True: rerun and overwrite existing seed results
OVERWRITE_RESULTS = False

# Continue other seeds if one seed fails
CONTINUE_ON_ERROR = False


# ============================================================
# 7. Output file paths
# ============================================================

ALL_RECOMMENDATIONS_FILE = os.path.join(
    RESULT_DIR,
    'GP-UCB_R1_all_seeds.csv'
)

CV_FOLD_FILE = os.path.join(
    RESULT_DIR,
    'GP-UCB_R1_CV_fold_results.csv'
)

SEED_SUMMARY_FILE = os.path.join(
    RESULT_DIR,
    'GP-UCB_R1_seed_summary.csv'
)

OVERALL_SUMMARY_FILE = os.path.join(
    RESULT_DIR,
    'GP-UCB_R1_overall_summary.csv'
)


# ============================================================
# 8. Display settings
# ============================================================

print("=" * 70)
print("GP-UCB: 30-seed R1 active learning")
print("=" * 70)

print(f"Experimental dataset: {SEED_FILE}")
print(f"Candidate space: {CANDIDATE_FILE}")
print(f"Number of seeds: {N_SEEDS}")
print(f"Initial training samples: {INITIAL_TRAIN_SIZE}")
print(f"CV folds: {N_SPLITS}")
print(f"Maximum batch size: {TOP_N}")

print("\nAcquisition parameters:")
print(f"Kappa: {KAPPA}")
print(f"Diversity scale: {DIVERSITY_SCALE}")
print(f"Boundary penalty: {LAMBDA_PEN}")
print(f"Minimum distance fraction: {MIN_DISTANCE_FRAC}")
print(f"Acquisition clipping: {CLIP_PERCENTILES}")
print(f"Uncertainty cap: {SIGMA_CAP_MULTIPLIER} × median sigma")

In [ ]:
# ============================================================
# Cell 2. Data loading and reproducible initial partitioning
# ============================================================


# ============================================================
# 1. Load experimental dataset
# ============================================================

def load_seed_data():

    if not os.path.exists(SEED_FILE):
        raise FileNotFoundError(
            f"Experimental dataset not found: {SEED_FILE}"
        )

    df = pd.read_csv(SEED_FILE)

    required_columns = FEATURE_NAMES + [TARGET_NAME]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise KeyError(
            f"SEED.csv is missing columns: {missing}"
        )

    df = df.copy()

    df[FEATURE_NAMES + [TARGET_NAME]] = (
        df[FEATURE_NAMES + [TARGET_NAME]].apply(
            pd.to_numeric,
            errors='coerce'
        )
    )

    if df[FEATURE_NAMES + [TARGET_NAME]].isnull().any().any():
        raise ValueError(
            "Missing or non-numeric values detected in SEED.csv."
        )

    if not np.isfinite(
        df[FEATURE_NAMES + [TARGET_NAME]].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Non-finite values detected in SEED.csv."
        )

    if len(df) != 15:
        raise ValueError(
            f"Expected 15 experimental samples, but found {len(df)}."
        )

    if INITIAL_TRAIN_SIZE >= len(df):
        raise ValueError(
            "Initial training size must be smaller than dataset size."
        )

    print("\nExperimental dataset loaded.")
    print(f"Total measured formulations: {len(df)}")
    print(f"Initial training size per seed: {INITIAL_TRAIN_SIZE}")
    print(
        f"Remaining measured formulations per seed: "
        f"{len(df) - INITIAL_TRAIN_SIZE}"
    )

    return df.reset_index(drop=True)


# ============================================================
# 2. Load independent candidate space
# ============================================================

def load_candidate_space():

    if not os.path.exists(CANDIDATE_FILE):
        raise FileNotFoundError(
            f"Candidate space not found: {CANDIDATE_FILE}"
        )

    candidate_df = pd.read_csv(CANDIDATE_FILE)

    missing = [
        col for col in FEATURE_NAMES
        if col not in candidate_df.columns
    ]

    if missing:
        raise KeyError(
            f"Candidate space is missing features: {missing}"
        )

    candidate_df = candidate_df.copy()

    candidate_df[FEATURE_NAMES] = (
        candidate_df[FEATURE_NAMES].apply(
            pd.to_numeric,
            errors='coerce'
        )
    )

    if candidate_df[FEATURE_NAMES].isnull().any().any():
        raise ValueError(
            "Missing or non-numeric candidate features detected."
        )

    if not np.isfinite(
        candidate_df[FEATURE_NAMES].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Non-finite candidate features detected."
        )

    # Preserve original row index
    if 'candidate_id' in candidate_df.columns:
        raise ValueError(
            "The candidate file already contains candidate_id. "
            "Please rename the original column before running."
        )

    candidate_df.insert(
        0,
        'candidate_id',
        np.arange(len(candidate_df))
    )

    print("\nIndependent candidate space loaded.")
    print(f"Total candidate formulations: {len(candidate_df)}")

    return candidate_df.reset_index(drop=True)


# ============================================================
# 3. Generate reproducible initial training partitions
# ============================================================

def generate_seed_partitions(
    experimental_df,
    seed_list,
    initial_train_size=12
):

    n_samples = len(experimental_df)

    all_indices = np.arange(n_samples)

    partitions = {}

    seen_partitions = set()

    for seed in seed_list:

        rng = np.random.default_rng(seed)

        train_idx = rng.choice(
            all_indices,
            size=initial_train_size,
            replace=False
        )

        train_idx = np.sort(train_idx)

        partition_key = tuple(train_idx.tolist())

        if partition_key in seen_partitions:
            raise RuntimeError(
                f"Duplicate initial partition detected at seed {seed}. "
                "Please use a different seed sequence."
            )

        seen_partitions.add(partition_key)

        remaining_idx = np.setdiff1d(
            all_indices,
            train_idx
        )

        partitions[seed] = {
            'train_idx': train_idx,
            'remaining_idx': remaining_idx
        }

    print("\nInitial partitions generated.")

    print(f"Number of seeds: {len(partitions)}")
    print(f"Initial training samples: {initial_train_size}")

    print(
        f"Unique initial partitions: {len(seen_partitions)}"
    )

    return partitions


# ============================================================
# 4. Exclude all previously tested formulations
# ============================================================

def remove_all_tested_formulations(
    candidate_df,
    experimental_df
):

    candidate_X = candidate_df[
        FEATURE_NAMES
    ].to_numpy(dtype=float)

    experimental_X = experimental_df[
        FEATURE_NAMES
    ].to_numpy(dtype=float)

    duplicate_mask = np.zeros(
        len(candidate_X),
        dtype=bool
    )

    for i, x in enumerate(candidate_X):

        duplicate_mask[i] = np.any(
            np.all(
                np.isclose(
                    experimental_X,
                    x,
                    rtol=1e-8,
                    atol=1e-10
                ),
                axis=1
            )
        )

    n_duplicate = int(
        np.sum(duplicate_mask)
    )

    remaining_df = (
        candidate_df.loc[~duplicate_mask]
        .copy()
        .reset_index(drop=True)
    )

    print("\nCandidate filtering completed.")

    print(f"Original candidate number: {len(candidate_df)}")
    print(f"Previously tested formulations: {n_duplicate}")
    print(f"Available candidates: {len(remaining_df)}")

    if remaining_df.empty:
        raise ValueError(
            "No untested candidates remain."
        )

    return remaining_df


# ============================================================
# 5. Prepare datasets
# ============================================================

experimental_df = load_seed_data()

candidate_df_original = load_candidate_space()

# Remove all 15 measured formulations from the candidate pool
candidate_space = remove_all_tested_formulations(
    candidate_df_original,
    experimental_df
)

# Generate 30 reproducible partitions
seed_partitions = generate_seed_partitions(
    experimental_df,
    SEED_LIST,
    INITIAL_TRAIN_SIZE
)

print("\nData preparation completed.")

In [ ]:
# ============================================================
# Cell 3. GP training and CV result extraction
# ============================================================


# ============================================================
# 1. Extract fold-level CV results when available
# ============================================================

def extract_fold_cv_results(optimizer, seed):

    """
    Extract genuine fold-level training and validation RMSE.

    This function does not reconstruct or fabricate fold-level
    results from the overall CV RMSE.
    """

    fold_df = pd.DataFrame(
        columns=[
            'seed',
            'iteration',
            'fold',
            'training_RMSE',
            'validation_RMSE'
        ]
    )


    attributes = [
        'fold_results_',
        'cv_fold_results_',
        'fold_metrics_',
        'cv_results_by_fold_',
        'fold_scores_'
    ]

    for attr in attributes:

        if not hasattr(optimizer, attr):
            continue

        raw = getattr(optimizer, attr)

        if isinstance(raw, pd.DataFrame):

            temp = raw.copy()

        elif isinstance(raw, (list, tuple)):

            try:
                temp = pd.DataFrame(raw)
            except Exception:
                continue

        elif isinstance(raw, dict):

            try:
                temp = pd.DataFrame(raw)
            except Exception:
                continue

        else:
            continue

        # Identify actual metric columns
        lower_map = {
            str(col).lower(): col
            for col in temp.columns
        }

        train_col = next(
            (
                lower_map[k]
                for k in lower_map
                if 'train' in k and 'rmse' in k
            ),
            None
        )

        val_col = next(
            (
                lower_map[k]
                for k in lower_map
                if ('val' in k or 'valid' in k)
                and 'rmse' in k
            ),
            None
        )

        if train_col is None or val_col is None:
            continue

        if len(temp) != N_SPLITS:
            continue

        fold_df = pd.DataFrame({
            'seed': seed,
            'iteration': 'R1',
            'fold': np.arange(1, N_SPLITS + 1),
            'training_RMSE': pd.to_numeric(
                temp[train_col],
                errors='coerce'
            ).to_numpy(),
            'validation_RMSE': pd.to_numeric(
                temp[val_col],
                errors='coerce'
            ).to_numpy()
        })

        if (
            fold_df[
                ['training_RMSE', 'validation_RMSE']
            ].notna().all().all()
        ):

            print(
                f"Fold-level CV results extracted from: {attr}"
            )

            return fold_df

    print(
        f"Seed {seed}: optimizer does not expose "
        "verifiable five-fold RMSE records through the "
        "checked attributes."
    )

    return fold_df


# ============================================================
# 2. Train GP model for one seed
# ============================================================

def train_gp_model(
    X_train,
    y_train,
    seed,
    n_splits=5,
    max_evals=50
):

    X_train = np.asarray(
        X_train,
        dtype=float
    )

    y_train = np.asarray(
        y_train,
        dtype=float
    ).ravel()

    if X_train.ndim != 2:
        raise ValueError(
            "X_train must be a 2D array."
        )

    if len(X_train) != len(y_train):
        raise ValueError(
            "X_train and y_train lengths do not match."
        )

    if len(y_train) < n_splits:
        raise ValueError(
            "Insufficient training samples for five-fold CV."
        )

    # Objective: maxmize RE
    y_train_loss = y_train

    optimizer = GPCrossValidatedOptimizer(
        X_train,
        y_train_loss,
        n_splits=n_splits,
        random_state=seed
    )

    optimizer.optimize(
        max_evals=max_evals
    )

    # Best observed objective value on the loss scale
    optimizer._best_value_ = float(
        np.min(y_train_loss)
    )

    optimizer.X_train_scaled_ = X_train.copy()

    # Overall CV metrics
    train_rmse = getattr(
        optimizer,
        'best_cv_train_rmse_',
        np.nan
    )

    val_rmse = getattr(
        optimizer,
        'best_cv_val_rmse_',
        np.nan
    )

    # Genuine fold-level metrics, if exposed
    fold_df = extract_fold_cv_results(
        optimizer,
        seed
    )

    model_info = {
        'seed': seed,
        'n_training_samples': len(y_train),
        'initial_best_RE': float(np.max(y_train)),
        'overall_training_RMSE': float(train_rmse),
        'overall_validation_RMSE': float(val_rmse),
        'best_params': str(
            getattr(optimizer, 'best_params', None)
        ),
        'fold_level_available': not fold_df.empty,
        'optimizer_random_state_argument': seed
    }

    return optimizer, fold_df, model_info

In [ ]:
# ============================================================
# Cell 4. UCB acquisition + diversity + boundary penalty
# ============================================================


def recommend_acquisition_experiments(
    candidate_df,
    optimizer,
    scaler,
    feature_names,
    X_train_scaled,
    top_n=5,
    kappa=0.5,
    diversity_scale=0.6,
    min_distance_frac=0.02,
    lambda_pen=0.4,
    sigma_cap_multiplier=3.0,
    clip_percentiles=(2, 98)
):

    EPS = 1e-12

    df = candidate_df.copy().reset_index(drop=True)

    # --------------------------------------------------------
    # 1. Input validation
    # --------------------------------------------------------

    if len(df) == 0:
        raise ValueError("Candidate pool is empty.")

    top_n = min(int(top_n), len(df))

    if top_n < 1:
        raise ValueError("top_n must be >= 1.")

    if kappa < 0:
        raise ValueError("kappa must be non-negative.")

    if not 0 <= diversity_scale <= 1:
        raise ValueError("Invalid diversity_scale.")

    if not 0 <= lambda_pen <= 1:
        raise ValueError("Invalid lambda_pen.")

    # --------------------------------------------------------
    # 2. Feature scaling
    # --------------------------------------------------------

    X_raw = df[feature_names].to_numpy(dtype=float)

    X = scaler.transform(X_raw)

    X_train_scaled = np.asarray(
        X_train_scaled,
        dtype=float
    )

    if len(X_train_scaled) == 0:
        raise ValueError("Training set is empty.")

    if X.shape[1] != X_train_scaled.shape[1]:
        raise ValueError("Feature dimensions do not match.")

    # --------------------------------------------------------
    # 3. GP predictions
    # --------------------------------------------------------

    mu, var = optimizer.predict(X)

    mu = np.asarray(mu, dtype=float).reshape(-1)
    var = np.asarray(var, dtype=float).reshape(-1)

    if len(mu) != len(X) or len(var) != len(X):
        raise ValueError("GP prediction dimensions mismatch.")

    if not np.isfinite(mu).all():
        raise ValueError("Non-finite prediction means.")

    if not np.isfinite(var).all():
        raise ValueError("Non-finite prediction variances.")

    # Current assumption: predict() returns variance
    sigma_raw = np.sqrt(
        np.maximum(var, 0.0)
    )

    # --------------------------------------------------------
    # 4. Uncertainty cap
    # --------------------------------------------------------

    sigma_median = np.median(sigma_raw)

    sigma_cap = (
        sigma_cap_multiplier * sigma_median
    )

    sigma = np.minimum(
        sigma_raw,
        sigma_cap
    )

    # --------------------------------------------------------
    # 5. Strict UCB acquisition
    # --------------------------------------------------------

    # GP training target = RE
    # acq(x) = mu(x) + kappa * sigma(x)

    acq = (
        mu + kappa * sigma
    )

    # --------------------------------------------------------
    # 6. Acquisition clipping
    # --------------------------------------------------------

    lower, upper = np.percentile(
        acq,
        clip_percentiles
    )

    acq_clipped = np.clip(
        acq,
        lower,
        upper
    )

    # --------------------------------------------------------
    # 7. Min-max normalization
    # --------------------------------------------------------

    acq_range = (
        np.max(acq_clipped)
        - np.min(acq_clipped)
    )

    if acq_range <= EPS:

        acq_norm = np.zeros_like(acq_clipped)

    else:

        acq_norm = (
            acq_clipped - np.min(acq_clipped)
        ) / acq_range

    # --------------------------------------------------------
    # 8. Distance-based diversity
    # --------------------------------------------------------

    distances = pairwise_distances(
        X,
        X_train_scaled
    )

    min_distance = np.min(
        distances,
        axis=1
    )

    distance_range = (
        np.max(min_distance)
        - np.min(min_distance)
    )

    if distance_range <= EPS:

        dist_norm = np.zeros_like(min_distance)

    else:

        dist_norm = (
            min_distance - np.min(min_distance)
        ) / distance_range

    # --------------------------------------------------------
    # 9. Boundary penalty
    # --------------------------------------------------------

    train_min = np.min(
        X_train_scaled,
        axis=0
    )

    train_max = np.max(
        X_train_scaled,
        axis=0
    )

    train_range = (
        train_max - train_min
    )

    train_range = np.where(
        train_range < EPS,
        EPS,
        train_range
    )

    d_left = (
        X - train_min
    ) / train_range

    d_right = (
        train_max - X
    ) / train_range

    near_boundary = np.minimum(
        d_left,
        d_right
    )

    near_boundary = np.clip(
        near_boundary,
        0.0,
        0.5
    )

    boundary_score = np.mean(
        near_boundary,
        axis=1
    )

    boundary_penalty = np.clip(
        1.0 - 2.0 * boundary_score,
        0.0,
        1.0
    )

    # --------------------------------------------------------
    # 10. Composite score
    # --------------------------------------------------------

    score1 = (
        (1.0 - diversity_scale) * acq_norm
        + diversity_scale * dist_norm
    )

    score = score1 * (
        1.0 - lambda_pen * boundary_penalty
    )

    # --------------------------------------------------------
    # 11. Minimum pairwise distance
    # --------------------------------------------------------

    ddiag = np.linalg.norm(
        np.max(X, axis=0)
        - np.min(X, axis=0)
    )

    dmin = (
        min_distance_frac * ddiag
    )

    # --------------------------------------------------------
    # 12. Strict batch selection
    # --------------------------------------------------------

    order = np.argsort(score)[::-1]

    selected = []

    for idx in order:

        if len(selected) >= top_n:
            break

        idx = int(idx)

        if len(selected) == 0:

            selected.append(idx)
            continue

        distances_to_selected = pairwise_distances(
            X[[idx]],
            X[selected]
        ).ravel()

        if np.all(
            distances_to_selected >= dmin
        ):

            selected.append(idx)

    selected = np.asarray(
        selected,
        dtype=int
    )

    if len(selected) == 0:
        raise RuntimeError(
            "No candidates satisfy the minimum distance constraint."
        )

    # --------------------------------------------------------
    # 13. Recommended formulations
    # --------------------------------------------------------

    recommended_df = (
        df.iloc[selected]
        .copy()
        .reset_index(drop=True)
    )

    recommended_df['predicted_loss'] = mu[selected]

    recommended_df['predicted_RE'] = -mu[selected]

    recommended_df['sigma_raw'] = sigma_raw[selected]

    recommended_df['sigma_used'] = sigma[selected]

    recommended_df['UCB_acquisition'] = acq[selected]

    recommended_df['acq_norm'] = acq_norm[selected]

    recommended_df['dist_norm'] = dist_norm[selected]

    recommended_df['boundary_penalty'] = (
        boundary_penalty[selected]
    )

    recommended_df['score1'] = score1[selected]

    recommended_df['final_score'] = score[selected]

    recommended_df['distance_to_training'] = (
        min_distance[selected]
    )

    recommended_df['dmin'] = dmin

    # --------------------------------------------------------
    # 14. Diagnostics
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("GP-UCB batch recommendation")
    print("=" * 65)

    print(f"Available candidates: {len(df)}")
    print(f"Selected candidates: {len(selected)}")

    print(f"Kappa: {kappa}")
    print(f"Diversity scale: {diversity_scale}")
    print(f"Boundary penalty: {lambda_pen}")

    print(f"Median sigma: {sigma_median:.6f}")
    print(f"Sigma cap: {sigma_cap:.6f}")

    print(f"Design-space diagonal: {ddiag:.6f}")
    print(f"Minimum distance: {dmin:.6f}")

    if len(selected) < top_n:
        print(
            "Warning: Fewer than TOP_N candidates satisfy dmin."
        )

    return selected, recommended_df

In [ ]:
# ============================================================
# Cell 5. Run 30 seeds for GP-UCB
# ============================================================


# ============================================================
# 1. Initialize result containers
# ============================================================

all_recommendations = []

all_fold_results = []

all_seed_summaries = []

failed_seeds = []


# ============================================================
# 2. Check existing results
# ============================================================

if not OVERWRITE_RESULTS:

    existing_seed_dirs = [
        os.path.join(
            RESULT_DIR,
            f"seed_{seed:02d}"
        )
        for seed in SEED_LIST
        if os.path.exists(
            os.path.join(
                RESULT_DIR,
                f"seed_{seed:02d}"
            )
        )
    ]

    if existing_seed_dirs:

        raise FileExistsError(
            "Existing seed result directories detected.\n"
            "Please check previous results or set "
            "OVERWRITE_RESULTS=True."
        )


# ============================================================
# 3. Main repeated-run loop
# ============================================================

for seed in SEED_LIST:

    print("\n\n" + "#" * 75)
    print(f"Running GP-UCB | Seed {seed:02d}")
    print("#" * 75)

    seed_dir = os.path.join(
        RESULT_DIR,
        f"seed_{seed:02d}"
    )

    os.makedirs(
        seed_dir,
        exist_ok=True
    )

    try:

        # ----------------------------------------------------
        # Step 1. Retrieve initial training indices
        # ----------------------------------------------------

        train_idx = seed_partitions[seed]['train_idx']

        remaining_idx = (
            seed_partitions[seed]['remaining_idx']
        )

        seed_train_df = (
            experimental_df.iloc[train_idx]
            .copy()
            .reset_index(drop=True)
        )

        seed_remaining_df = (
            experimental_df.iloc[remaining_idx]
            .copy()
            .reset_index(drop=True)
        )

        # Preserve original row indices
        seed_train_df.insert(
            0,
            'source_row_index',
            train_idx
        )

        seed_train_df.insert(
            0,
            'seed',
            seed
        )

        seed_remaining_df.insert(
            0,
            'source_row_index',
            remaining_idx
        )

        # ----------------------------------------------------
        # Step 2. Save initial training data
        # ----------------------------------------------------

        initial_file = os.path.join(
            seed_dir,
            'R1_initial_training_data.csv'
        )

        seed_train_df.to_csv(
            initial_file,
            index=False,
            encoding='utf-8-sig'
        )

        # ----------------------------------------------------
        # Step 3. Prepare training data
        # ----------------------------------------------------

        X_train_raw = (
            experimental_df.iloc[train_idx][
                FEATURE_NAMES
            ].to_numpy(dtype=float)
        )

        y_train = (
            experimental_df.iloc[train_idx][
                TARGET_NAME
            ].to_numpy(dtype=float)
        )

        # Fit scaler using only 12 training samples
        scaler = StandardScaler()

        X_train = scaler.fit_transform(
            X_train_raw
        )

        # Transform independent candidate space
        X_candidate = scaler.transform(
            candidate_space[FEATURE_NAMES].to_numpy(
                dtype=float
            )
        )

        print(f"Seed: {seed}")
        print(f"Initial training samples: {len(X_train)}")
        print(f"Remaining measured samples: {len(remaining_idx)}")
        print(f"Available candidates: {len(X_candidate)}")
        print(f"Initial best RE: {np.max(y_train):.4f}")

        # ----------------------------------------------------
        # Step 4. Train GP model
        # ----------------------------------------------------

        optimizer, fold_df, model_info = train_gp_model(
            X_train=X_train,
            y_train=y_train,
            seed=seed,
            n_splits=N_SPLITS,
            max_evals=MAX_EVALS
        )

        # ----------------------------------------------------
        # Step 5. Recommend R1 formulations
        # ----------------------------------------------------

        selected_indices, recommended_df = (
            recommend_acquisition_experiments(

                candidate_df=candidate_space,

                optimizer=optimizer,

                scaler=scaler,

                feature_names=FEATURE_NAMES,

                X_train_scaled=X_train,

                top_n=TOP_N,

                kappa=KAPPA,

                diversity_scale=DIVERSITY_SCALE,

                min_distance_frac=MIN_DISTANCE_FRAC,

                lambda_pen=LAMBDA_PEN,

                sigma_cap_multiplier=SIGMA_CAP_MULTIPLIER,

                clip_percentiles=CLIP_PERCENTILES

            )
        )

        # ----------------------------------------------------
        # Step 6. Add seed information
        # ----------------------------------------------------

        recommended_df.insert(
            0,
            'seed',
            seed
        )

        recommended_df.insert(
            1,
            'round',
            'R1'
        )

        # ----------------------------------------------------
        # Step 7. Save recommendations
        # ----------------------------------------------------

        recommendation_file = os.path.join(
            seed_dir,
            'R1_recommended.csv'
        )

        recommended_df.to_csv(
            recommendation_file,
            index=False,
            encoding='utf-8-sig'
        )

        all_recommendations.append(
            recommended_df.copy()
        )

        # ----------------------------------------------------
        # Step 8. Save CV fold results
        # ----------------------------------------------------

        fold_file = os.path.join(
            seed_dir,
            'R1_CV_results.csv'
        )

        fold_df.to_csv(
            fold_file,
            index=False,
            encoding='utf-8-sig'
        )

        if not fold_df.empty:

            all_fold_results.append(
                fold_df.copy()
            )

        # ----------------------------------------------------
        # Step 9. Save model information
        # ----------------------------------------------------

        model_info.update({

            'seed': seed,

            'n_initial_samples': len(train_idx),

            'n_remaining_measured_samples': len(remaining_idx),

            'n_available_candidates': len(candidate_space),

            'n_recommended': len(recommended_df),

            'KAPPA': KAPPA,

            'DIVERSITY_SCALE': DIVERSITY_SCALE,

            'LAMBDA_PEN': LAMBDA_PEN,

            'MIN_DISTANCE_FRAC': MIN_DISTANCE_FRAC,

            'CLIP_PERCENTILES': str(CLIP_PERCENTILES),

            'SIGMA_CAP_MULTIPLIER': SIGMA_CAP_MULTIPLIER,

            'TOP_N': TOP_N
        })

        model_info_file = os.path.join(
            seed_dir,
            'R1_model_info.txt'
        )

        with open(
            model_info_file,
            'w',
            encoding='utf-8'
        ) as f:

            for key, value in model_info.items():

                f.write(
                    f"{key}: {value}\n"
                )

        # ----------------------------------------------------
        # Step 10. Save seed-level summary
        # ----------------------------------------------------

        seed_summary = {

            'seed': seed,

            'n_initial_samples': len(train_idx),

            'n_recommended': len(recommended_df),

            'initial_best_RE': float(np.max(y_train)),

            'initial_mean_RE': float(np.mean(y_train)),

            'training_RMSE': model_info[
                'overall_training_RMSE'
            ],

            'validation_RMSE': model_info[
                'overall_validation_RMSE'
            ],

            'fold_level_available': model_info[
                'fold_level_available'
            ],

            'best_params': model_info['best_params'],

            'status': 'Success'
        }

        all_seed_summaries.append(
            seed_summary
        )

        print(
            f"\nSeed {seed:02d} completed successfully."
        )

        print(
            f"Recommended formulations: {len(recommended_df)}"
        )

    except Exception as e:

        error_message = traceback.format_exc()

        failed_seeds.append({

            'seed': seed,

            'error': str(e),

            'traceback': error_message

        })

        print(f"\nSeed {seed:02d} failed.")
        print(error_message)

        if not CONTINUE_ON_ERROR:
            raise

        all_seed_summaries.append({

            'seed': seed,

            'n_initial_samples': INITIAL_TRAIN_SIZE,

            'n_recommended': 0,

            'initial_best_RE': np.nan,

            'initial_mean_RE': np.nan,

            'training_RMSE': np.nan,

            'validation_RMSE': np.nan,

            'fold_level_available': False,

            'best_params': None,

            'status': 'Failed'

        })


# ============================================================
# 4. Save combined recommendation results
# ============================================================

if all_recommendations:

    all_recommendations_df = pd.concat(
        all_recommendations,
        ignore_index=True
    )

    all_recommendations_df.to_csv(
        ALL_RECOMMENDATIONS_FILE,
        index=False,
        encoding='utf-8-sig'
    )

    print("\nAll recommendation results saved:")
    print(ALL_RECOMMENDATIONS_FILE)


# ============================================================
# 5. Save combined fold-level results
# ============================================================

fold_columns = [
    'seed',
    'iteration',
    'fold',
    'training_RMSE',
    'validation_RMSE'
]

if all_fold_results:

    all_fold_df = pd.concat(
        all_fold_results,
        ignore_index=True
    )

else:

    all_fold_df = pd.DataFrame(
        columns=fold_columns
    )

all_fold_df.to_csv(
    CV_FOLD_FILE,
    index=False,
    encoding='utf-8-sig'
)

print("\nFold-level CV results saved:")
print(CV_FOLD_FILE)

if all_fold_df.empty:

    print(
        "WARNING: No verifiable fold-level CV records "
        "were exposed by the optimizer."
    )


# ============================================================
# 6. Save seed-level summary
# ============================================================

seed_summary_df = pd.DataFrame(
    all_seed_summaries
)

seed_summary_df.to_csv(
    SEED_SUMMARY_FILE,
    index=False,
    encoding='utf-8-sig'
)

print("\nSeed-level summary saved:")
print(SEED_SUMMARY_FILE)


# ============================================================
# 7. Save failed seeds
# ============================================================

if failed_seeds:

    failed_df = pd.DataFrame([
        {
            'seed': item['seed'],
            'error': item['error']
        }
        for item in failed_seeds
    ])

    failed_df.to_csv(
        os.path.join(
            RESULT_DIR,
            'GP-UCB_R1_failed_seeds.csv'
        ),
        index=False,
        encoding='utf-8-sig'
    )


print("\n" + "=" * 75)
print("GP-UCB 30-seed execution finished.")
print("=" * 75)

print(f"Successful seeds: {len(all_seed_summaries) - len(failed_seeds)}")
print(f"Failed seeds: {len(failed_seeds)}")

In [ ]:
# ============================================================
# Cell 6. Seed-level and across-seed statistical summaries
# ============================================================


# ============================================================
# 1. Reload seed-level summary
# ============================================================

if not os.path.exists(SEED_SUMMARY_FILE):
    raise FileNotFoundError(
        "Seed summary not found. Please run Cell 5 first."
    )

seed_summary_df = pd.read_csv(
    SEED_SUMMARY_FILE
)

successful_df = seed_summary_df[
    seed_summary_df['status'] == 'Success'
].copy()


# ============================================================
# 2. Calculate seed-level CV statistics
# ============================================================

cv_metric_columns = [
    'training_RMSE',
    'validation_RMSE'
]

seed_level_stats = []

for metric in cv_metric_columns:

    values = pd.to_numeric(
        successful_df[metric],
        errors='coerce'
    ).dropna()

    seed_level_stats.append({

        'metric': metric,

        'n_seeds': len(values),

        'mean': values.mean() if len(values) else np.nan,

        'SD': values.std(ddof=1) if len(values) > 1 else np.nan,

        'minimum': values.min() if len(values) else np.nan,

        'maximum': values.max() if len(values) else np.nan

    })


# ============================================================
# 3. Calculate recommendation statistics
# ============================================================

if os.path.exists(ALL_RECOMMENDATIONS_FILE):

    all_rec_df = pd.read_csv(
        ALL_RECOMMENDATIONS_FILE
    )

    recommendation_stats = {

        'total_recommendations': len(all_rec_df),

        'successful_seed_count': len(successful_df),

        'mean_recommendations_per_seed': (
            successful_df['n_recommended'].mean()
        ),

        'maximum_recommendations_per_seed': (
            successful_df['n_recommended'].max()
        ),

        'minimum_recommendations_per_seed': (
            successful_df['n_recommended'].min()
        )

    }

else:

    recommendation_stats = {

        'total_recommendations': 0,

        'successful_seed_count': len(successful_df),

        'mean_recommendations_per_seed': np.nan,

        'maximum_recommendations_per_seed': np.nan,

        'minimum_recommendations_per_seed': np.nan

    }


# ============================================================
# 4. Calculate fold-level statistics if available
# ============================================================

fold_summary_rows = []

if os.path.exists(CV_FOLD_FILE):

    fold_data = pd.read_csv(
        CV_FOLD_FILE
    )

    if not fold_data.empty:

        for fold in sorted(
            fold_data['fold'].unique()
        ):

            fold_subset = fold_data[
                fold_data['fold'] == fold
            ]

            for metric in [
                'training_RMSE',
                'validation_RMSE'
            ]:

                values = pd.to_numeric(
                    fold_subset[metric],
                    errors='coerce'
                ).dropna()

                fold_summary_rows.append({

                    'fold': fold,

                    'metric': metric,

                    'n_seeds': len(values),

                    'mean_across_seeds': (
                        values.mean()
                        if len(values) else np.nan
                    ),

                    'SD_across_seeds': (
                        values.std(ddof=1)
                        if len(values) > 1 else np.nan
                    )

                })

fold_summary_df = pd.DataFrame(
    fold_summary_rows
)


# ============================================================
# 5. Save overall summary
# ============================================================

overall_rows = []

for item in seed_level_stats:

    overall_rows.append({

        'category': 'Seed-level CV',

        'metric': item['metric'],

        'n': item['n_seeds'],

        'mean': item['mean'],

        'SD': item['SD'],

        'minimum': item['minimum'],

        'maximum': item['maximum']

    })


for key, value in recommendation_stats.items():

    overall_rows.append({

        'category': 'Recommendation',

        'metric': key,

        'n': len(successful_df),

        'mean': value,

        'SD': np.nan,

        'minimum': np.nan,

        'maximum': np.nan

    })


overall_summary_df = pd.DataFrame(
    overall_rows
)

overall_summary_df.to_csv(
    OVERALL_SUMMARY_FILE,
    index=False,
    encoding='utf-8-sig'
)


# ============================================================
# 6. Display summaries
# ============================================================

print("\n" + "=" * 70)
print("Seed-level summary")
print("=" * 70)

display(
    successful_df[
        [
            'seed',
            'n_initial_samples',
            'n_recommended',
            'initial_best_RE',
            'training_RMSE',
            'validation_RMSE',
            'fold_level_available',
            'status'
        ]
    ]
)

print("\n" + "=" * 70)
print("Across-seed summary")
print("=" * 70)

display(overall_summary_df)

if not fold_summary_df.empty:

    print("\nFold-level summary across seeds:")

    display(fold_summary_df)

else:

    print(
        "\nFold-level summary unavailable: "
        "the optimizer did not expose verifiable fold-level records."
    )

print("\nOverall summary saved:")
print(OVERALL_SUMMARY_FILE)

In [ ]:
# ============================================================
# Cell 7. Repeated-run integrity checks
# ============================================================


# ============================================================
# 1. Check seed completion
# ============================================================

if not os.path.exists(SEED_SUMMARY_FILE):

    raise FileNotFoundError(
        "Seed summary file is missing."
    )

summary_df = pd.read_csv(
    SEED_SUMMARY_FILE
)

successful = summary_df[
    summary_df['status'] == 'Success'
]

failed = summary_df[
    summary_df['status'] == 'Failed'
]

print("\n" + "=" * 70)
print("30-seed integrity check")
print("=" * 70)

print(f"Expected seeds: {N_SEEDS}")
print(f"Completed seeds: {len(successful)}")
print(f"Failed seeds: {len(failed)}")

if len(summary_df) != N_SEEDS:
    print("WARNING: Seed count does not equal 30.")

if len(successful) != N_SEEDS:
    print(
        "WARNING: Not all 30 seeds completed successfully."
    )


# ============================================================
# 2. Check initial training sample size
# ============================================================

initial_sizes = []

for seed in SEED_LIST:

    initial_file = os.path.join(
        RESULT_DIR,
        f"seed_{seed:02d}",
        "R1_initial_training_data.csv"
    )

    if not os.path.exists(initial_file):
        print(
            f"WARNING: Missing initial data for seed {seed}."
        )
        continue

    initial_df = pd.read_csv(
        initial_file
    )

    initial_sizes.append(
        len(initial_df)
    )

    if len(initial_df) != INITIAL_TRAIN_SIZE:

        print(
            f"WARNING: Seed {seed} has "
            f"{len(initial_df)} initial samples."
        )

print("\nInitial training sample sizes:")
print(pd.Series(initial_sizes).value_counts())


# ============================================================
# 3. Check unique initial partitions
# ============================================================

partition_keys = []

for seed in SEED_LIST:

    if seed not in seed_partitions:
        continue

    key = tuple(
        seed_partitions[seed]['train_idx'].tolist()
    )

    partition_keys.append(key)

n_unique_partitions = len(
    set(partition_keys)
)

print("\nUnique initial partitions:")
print(
    f"{n_unique_partitions}/{len(SEED_LIST)}"
)

if n_unique_partitions != len(SEED_LIST):

    print(
        "WARNING: Duplicate initial partitions detected."
    )


# ============================================================
# 4. Check recommendation files
# ============================================================

recommendation_files = []

for seed in SEED_LIST:

    file_path = os.path.join(
        RESULT_DIR,
        f"seed_{seed:02d}",
        "R1_recommended.csv"
    )

    if os.path.exists(file_path):

        recommendation_files.append(
            file_path
        )

print("\nRecommendation files:")
print(
    f"{len(recommendation_files)}/{N_SEEDS}"
)


# ============================================================
# 5. Check repeated recommendations across seeds
# ============================================================

if os.path.exists(ALL_RECOMMENDATIONS_FILE):

    recommendations = pd.read_csv(
        ALL_RECOMMENDATIONS_FILE
    )

    recommendations['formulation_key'] = (
        recommendations[FEATURE_NAMES]
        .round(10)
        .astype(str)
        .agg('|'.join, axis=1)
    )

    duplicate_recommendations = (
        recommendations.groupby(
            ['seed', 'formulation_key']
        ).size()
    )

    within_seed_duplicates = (
        duplicate_recommendations > 1
    ).sum()

    print("\nRepeated formulations within the same seed:")
    print(within_seed_duplicates)

    # Cross-seed overlap is allowed:
    # different seeds may recommend the same candidate.
    print(
        "\nCross-seed repeated recommendations are allowed "
        "and are not treated as errors."
    )


# ============================================================
# 6. Check overlap with the 15 measured formulations
# ============================================================

if os.path.exists(ALL_RECOMMENDATIONS_FILE):

    recommendations = pd.read_csv(
        ALL_RECOMMENDATIONS_FILE
    )

    tested_X = experimental_df[
        FEATURE_NAMES
    ].to_numpy(dtype=float)

    rec_X = recommendations[
        FEATURE_NAMES
    ].to_numpy(dtype=float)

    overlap_count = 0

    for x in rec_X:

        if np.any(
            np.all(
                np.isclose(
                    tested_X,
                    x,
                    rtol=1e-8,
                    atol=1e-10
                ),
                axis=1
            )
        ):

            overlap_count += 1

    print("\nOverlap with existing measured formulations:")
    print(overlap_count)

    if overlap_count > 0:

        print(
            "WARNING: Recommended candidates overlap "
            "with previously tested formulations."
        )


# ============================================================
# 7. Check measured RE handling
# ============================================================

if os.path.exists(ALL_RECOMMENDATIONS_FILE):

    recommendations = pd.read_csv(
        ALL_RECOMMENDATIONS_FILE
    )

    if 'RE' in recommendations.columns:

        raise ValueError(
            "The recommendation file contains an RE column. "
            "Predicted RE must not be treated as measured RE."
        )

    print(
        "\nMeasured RE handling check passed: "
        "no experimental RE was fabricated."
    )


# ============================================================
# 8. Final status
# ============================================================

print("\n" + "=" * 70)
print("Integrity check completed.")
print("=" * 70)